# Customer Churn Prediction — Logistic Regression

This notebook is a cleaned, reproducible version of the Churn section of the original model notebook.

**Workflow**
1. Load the Churn dataset
2. Inspect and clean the data
3. Convert `TotalCharges` to numeric
4. Encode categorical variables
5. Train/test split
6. Train Logistic Regression
7. Evaluate Accuracy, Precision, Recall, F1 and ROC-AUC
8. Plot the ROC curve
9. Save the trained preprocessing + model pipeline for Streamlit

> Run the notebook from top to bottom. The dataset is loaded from `../data/Churn.csv`, so it works without changing a Windows file path.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve,
    classification_report
)
import joblib

pd.set_option("display.max_columns", None)


In [ ]:
DATA_PATH = os.path.join("..", "data", "Churn.csv")

if not os.path.exists(DATA_PATH):
    # Fallback for running the notebook from the project root.
    DATA_PATH = os.path.join("data", "Churn.csv")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


In [ ]:
print(df.info())
print("\nMissing values:")
display(df.isnull().sum())


In [ ]:
# Remove the customer identifier because it has no predictive meaning.
df = df.drop(columns=["customerID"])

# TotalCharges contains numeric values stored as text and some blank strings.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Churn is the target: Yes = 1, No = 0.
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Missing values after cleaning:")
display(df.isnull().sum())


In [ ]:
print("Churn distribution:")
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).mul(100).round(2))


In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=16729,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
print("Model training completed.")


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})
metrics["Score (%)"] = (metrics["Score"] * 100).round(2)
display(metrics)


In [ ]:
print("Confusion Matrix:")
display(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Customer Churn Prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Save the complete pipeline. This includes preprocessing + Logistic Regression.
MODEL_DIR = os.path.join("..", "models")
if not os.path.exists(MODEL_DIR):
    MODEL_DIR = os.path.join("models")

os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "churn_logistic_regression_pipeline.joblib")
joblib.dump(model, MODEL_PATH)

print("Saved model to:", MODEL_PATH)


In [ ]:
# Quick deployment test: reload exactly what Streamlit will use.
loaded_model = joblib.load(MODEL_PATH)
test_prob = loaded_model.predict_proba(X_test.iloc[:5])[:, 1]

display(pd.DataFrame({
    "Actual": y_test.iloc[:5].values,
    "Predicted": loaded_model.predict(X_test.iloc[:5]),
    "Churn Probability (%)": (test_prob * 100).round(2)
}))


## Conclusion

The model predicts whether a customer is likely to churn. The saved `.joblib` file contains the complete preprocessing and Logistic Regression pipeline, allowing the Streamlit application to use the same transformations as the notebook.

The notebook can be rerun from top to bottom whenever the dataset or model needs to be updated.
